# Lab 11: Model selection, regularization, priors

A model fits the data it was trained on. This lab is about choosing a model that predicts new data. There are three tools: held-out data (cross-validation), evidence (Bayes), and a penalty on complex models (regularization).

1. The hard function: polynomial degree 1, 2, 5, 20.
2. Bias and variance by simulation.
3. Cross-validation, and the leakage bug (with the first part of the computing corner: standardizing).
4. Choosing by evidence: the death-penalty data.
5. James-Stein: learning the prior from the data.
6. A prior is a regularizer: ridge and lasso (with the second part of the computing corner: solve, do not invert).
7. Loss as negative log-likelihood; softmax cross-entropy for K classes.

Then the exit problem on paper and the At home section.

Lab exercises are marked **(In class)** or **(At home)**. Fill in the empty code cell and run the check cell under it. Upload the notebook on Canvas with the outputs left in. Every random step has a fixed seed. Simulated data are called simulated.


### Route for this lab

**Three outcomes.** After this lab you can (1) choose a model by held-out data, with training-only preprocessing, and spot a leak; (2) say how a prior, a regularizer and a loss function are the same object seen three ways; (3) compute a softmax cross-entropy for K classes by hand and in code.

**In class (core sections).** Sections 1 to 3 (cross-validation and the leak), then 4 (evidence), 5 (James-Stein), 6 (ridge and lasso), 7 (loss and cross-entropy). The break comes after Section 3.

**Required.** All seven sections and the six graded exercises, 11.1 to 11.6. Exercises 11.1 to 11.5 are In class and sit right after the example they practise. Exercise 11.6 is At home. The Predict, Your turn and Check yourself items inside the sections are done in class and are not graded.

**Due.** Before class on 2 December. Submit the notebook on Canvas with the outputs left in.

Run All stops at the first empty exercise check; that is expected. Fill the exercises, then Restart and Run All before you submit.


### Online labs for this notebook

- [Overfitting Lab](https://vsokolov.org/labs/overfitting-lab.html): Sections 1 to 3. Training error keeps falling as the model grows, while the test error traces a U.
- [Calibration Lab](https://vsokolov.org/labs/calibration-lab.html): Section 3. Scoring a fitted model on the loans it was fitted to against new loans.
- [Needle in a Haystack](https://vsokolov.org/labs/sparsity-lab.html): Section 6. Least squares, ridge and the lasso on 1,000 genes and 100 patients.


In [ ]:
from hashlib import sha256
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy.polynomial import Polynomial
from scipy.special import betaln

DATA = Path("data")
DATA.mkdir(exist_ok=True)
BASE = "https://vsokolov.org/courses/files/600/data"
for name in ("credit.csv", "EfronMorrisBB.txt"):
    path = DATA / name
    if not path.exists():
        urlretrieve(f"{BASE}/{name}", path)

plt.rcParams["figure.figsize"] = (5, 3.5)


def check(name, value, expected, ndigits=6):
    if value is None:
        raise AssertionError(f"{name} is missing: run the exercise cell above and define {name}")
    try:
        v = float(value)
    except (TypeError, ValueError):
        raise AssertionError(f"{name} must be a single number, got {type(value).__name__}")
    if not np.isfinite(v):
        raise AssertionError(f"{name} is not a finite number (got {v}): look for a division by zero or a log of zero")
    s = f"{round(v, ndigits):.{ndigits}f}"
    got = sha256(f"aii600-lab11|{name}|{s}".encode()).hexdigest()[:16]
    assert got == expected, f"{name} = {v:.6f} does not match the reference value: re-read the task, the inputs and the formula"
    print("ok ", name)


def ridge_fit(X, y, lam, mu=None, sd=None):
    """Ridge on standardized features. mu and sd default to the mean and sd of the rows given."""
    mu = X.mean(axis=0) if mu is None else mu
    sd = X.std(axis=0) if sd is None else sd
    Z = (X - mu) / sd
    zbar = Z.mean(axis=0)
    Z = Z - zbar
    ybar = y.mean()
    beta = np.linalg.solve(Z.T @ Z + lam * np.eye(Z.shape[1]), Z.T @ (y - ybar))
    return mu, sd, zbar, ybar, beta


def ridge_predict(model, X):
    mu, sd, zbar, ybar, beta = model
    return ybar + ((X - mu) / sd - zbar) @ beta


## 1. The hard function

We want a model that predicts well on data it has not seen. The training error cannot tell us that, because a flexible model can fit the training points almost perfectly and still be wrong everywhere else.

Here is a test case from the book. The true curve is

$$
f(x) = \frac{1}{1 + 25 x^2}.
$$

The data are simulated: 40 values of $x$ drawn uniformly from $[-2, 2]$, and $y = f(x) + \varepsilon$, where the noise $\varepsilon$ has standard deviation 0.1. So the noise variance is 0.01. That is the smallest *expected* squared prediction error any model can have under this setup. We also simulate 500 fresh points as a test set. A test MSE computed from 500 points is one random number, so it can land above or below 0.01.

We fit a polynomial $\hat y = \beta_0 + \beta_1 x + \dots + \beta_d x^d$ of degree $d$. The degree is the model complexity. The error measure is the mean squared error (MSE): the average of $(y - \hat y)^2$.

**Online lab.** Open the [Overfitting Lab](https://vsokolov.org/labs/overfitting-lab.html) next to this section. Try this: raise the model complexity step by step and watch the training error fall while the test error first falls and then rises. It is the same U you will see in the table below.

**Predict.** Before you run the cell: which of the degrees 1, 2, 5, 10, 20 will have the lowest training error? Which will have the lowest test error? Write down both.


In [ ]:
f = lambda x: 1 / (1 + 25 * x**2)

rng = np.random.default_rng(3)
n = 40
x_tr = np.sort(rng.uniform(-2, 2, n))
y_tr = f(x_tr) + rng.normal(0, 0.1, n)
x_te = rng.uniform(-2, 2, 500)
y_te = f(x_te) + rng.normal(0, 0.1, 500)

rows = []
fits = {}
for d in (1, 2, 5, 10, 20):
    fits[d] = Polynomial.fit(x_tr, y_tr, d)
    rows.append((d, np.mean((fits[d](x_tr) - y_tr) ** 2), np.mean((fits[d](x_te) - y_te) ** 2)))
hard = pd.DataFrame(rows, columns=["degree", "train MSE", "test MSE"]).set_index("degree")
print(hard.round(4))

# the noise floor is an expectation: how often does the TRUE curve score below 0.01 on 500 test points?
rng_floor = np.random.default_rng(0)
floor_share = np.mean([np.mean(rng_floor.normal(0, 0.1, 500) ** 2) < 0.01 for _ in range(2000)])
print("true curve f: share of 2000 simulated test sets (500 points each) with MSE below 0.01:", round(floor_share, 3))

xs = np.linspace(-2, 2, 400)
fig, axes = plt.subplots(2, 3, figsize=(9, 5), sharex=True, sharey=True)
axes[1, 2].axis("off")
for ax, d in zip(axes.ravel(), (1, 2, 5, 10, 20)):
    ax.scatter(x_tr, y_tr, s=8, color="gray")
    ax.plot(xs, f(xs), color="black", lw=1, label="true f")
    ax.plot(xs, fits[d](xs), color="C3", ls="--", label=f"degree {d}")
    ax.set_ylim(-0.5, 1.5)
    ax.legend(fontsize=7, loc="upper right")
for ax in axes[1, :2]:
    ax.set_xlabel("x")
for ax in axes[:, 0]:
    ax.set_ylabel("y")
plt.tight_layout()
plt.show()


**Reading.**

- Training MSE only goes down as the degree goes up: 0.0530 for degree 1, 0.0065 for degree 20. Degree 20 has the lowest training error, far below the noise variance of 0.01. It has fitted the noise.
- Test MSE goes down and then up. Degree 10 has the lowest test MSE of the five, 0.0301. Degree 20 has a test MSE of 149.9. That is thousands of times worse.
- In the plot, the degree 1 line is too stiff (it cannot bend into the bump at 0). Degree 20 passes near every point but swings wildly between them and near the ends.
- Too little flexibility is called underfitting. Too much is overfitting. The training error cannot see the difference. Held-out data can.

**Check yourself.**

1. Degree 20 has the smallest training error. Why is that not a reason to choose it?
2. The noise variance is 0.01. Could a model have a test MSE below 0.01 on one test set?

<details><summary>Answer</summary>

1. Training error rewards fitting the noise. The model with 21 parameters has enough freedom to pass near all 40 points. Its test MSE shows it did not learn the curve.
2. Yes, it can happen. The best possible expected prediction MSE is 0.01 under this data-generating model. A finite test-set MSE fluctuates around that value and can be above or below it: even the true curve scored below 0.01 in 50% of the simulated test sets above. A score below 0.01 does not mean the model predicted the independent noise.

</details>


## 2. Bias and variance by simulation

The test error of a model has three parts. Take one new point $x_0$. The new response is $y_0 = f(x_0) + \varepsilon$. Our fitted model predicts $\hat y_0$. Then

$$
\mathbb{E}\,(y_0 - \hat y_0)^2 = \underbrace{\sigma^2}_{\text{noise}} + \underbrace{(f(x_0) - \mathbb{E}\,\hat y_0)^2}_{\text{bias squared}} + \underbrace{\mathrm{Var}(\hat y_0)}_{\text{variance}}.
$$

- $\sigma^2$ is the noise variance, 0.01. We cannot change it.
- Bias is how far the average prediction is from the truth. A model that is too stiff has high bias.
- Variance is how much the prediction moves from one training set to another. A model that is too flexible has high variance.

The expectation $\mathbb{E}$ means "average over many training sets". We do that by simulation. Everything below is simulated.

- Keep the same 40 values of $x$ as in Section 1 (`x_tr`). Draw fresh noise for $y$ 200 times and refit each degree each time. This gives 200 fitted curves per degree.
- Judge the predictions at the same 500 test locations `x_te` as in Section 1, for the same degrees 1, 2, 5, 10 and 20.
- The formula above is for one point $x_0$. The table reports bias squared and variance averaged over the 500 test locations.

There are two differences from Section 1. Section 1 used one training noise draw, and here we average over 200. Section 1 used one test response at each location, and here each of the 200 rounds draws a fresh one.


In [ ]:
x0 = np.sort(x_te)                   # where we judge the predictions: the Section 1 test locations
sigma = 0.1                          # change this: noise sd (try 0.3)
n_rep = 200
rng_bv = np.random.default_rng(5)
noise = rng_bv.normal(0, sigma, (n_rep, len(x_tr)))          # fresh noise for each refit, same x_tr
y_new = f(x0) + rng_bv.normal(0, sigma, (n_rep, len(x0)))    # fresh responses at x0

degrees_bv = (1, 2, 5, 10, 20)
preds = {}
for d in degrees_bv:
    preds[d] = np.array([Polynomial.fit(x_tr, f(x_tr) + noise[r], d)(x0) for r in range(n_rep)])

rows = []
for d in degrees_bv:
    bias2 = np.mean((preds[d].mean(axis=0) - f(x0)) ** 2)
    var = np.mean(preds[d].var(axis=0))
    measured = np.mean((y_new - preds[d]) ** 2)
    rows.append((d, bias2, var, sigma**2, bias2 + var + sigma**2, measured))
bv = pd.DataFrame(rows, columns=["degree", "bias^2", "variance", "noise", "total", "measured test MSE"]).set_index("degree")
bv["Section 1 (one fit)"] = hard["test MSE"]
print(bv.round(4))

fig, axes = plt.subplots(2, 3, figsize=(9, 5), sharex=True, sharey=True)
axes[1, 2].axis("off")
for ax, d in zip(axes.ravel(), degrees_bv):
    for r in range(30):
        ax.plot(x0, preds[d][r], color="C0", alpha=0.25, lw=0.8)
    ax.plot(x0, f(x0), color="black", ls="--", lw=1.5, label="true f")
    ax.set_ylim(-0.5, 1.5)
    ax.set_title(f"degree {d}: 30 of 200 fits", fontsize=9)
    ax.legend(fontsize=7, loc="upper right")
for ax in axes[1, :2]:
    ax.set_xlabel("x")
for ax in axes[:, 0]:
    ax.set_ylabel("predicted y")
plt.tight_layout()
plt.show()


**Reading.**

- Degree 1: bias squared is 0.0648 and variance is 0.0005. The 200 lines are tightly bunched and all miss the bump. High bias, low variance.
- Degree 2: bias squared 0.0453, variance 0.0008.
- Degree 5: bias squared 0.0316, variance 0.0017. Bias has fallen and variance has grown.
- Degree 10: bias squared 0.0105, variance 0.0049. This degree has the smallest total, 0.0254, of the five.
- Degree 20: bias squared 1.79, variance 54.8. The variance is about 11263 times that of degree 10, and the total is 56.6. The 200 curves in the plot swing wildly, especially near the ends. Too flexible: the variance is the problem.
- Total (noise plus bias squared plus variance) and "measured test MSE" (fresh responses against the fitted curves) agree up to simulation error, as the formula says.
- Compare the last two columns. Section 1 (one training set, same test locations) gave 0.0301 for degree 10 and 149.9 for degree 20. The average over 200 training noise draws gives 0.0253 and 56.6. The degree ordering is the same, and the decomposition says where the error comes from: degree 20 is all variance. The single fit of Section 1 is one draw from this spread.

**Your turn.** Change `sigma` to 0.3 in the cell above (the line has a comment) and run it again. Target: say which degree has the smallest "total" now, and check that the variance column for degree 5 became 9 times larger (the noise sd tripled and a variance grows with the square of it).

**Check yourself.**

1. In the table, which of the two terms, bias squared or variance, grows with the degree? Which one falls, and up to which degree?
2. Why do we need many training sets to see variance? In practice we have only one. What do we use instead?

<details><summary>Answer</summary>

1. Variance grows with the degree. Bias squared falls up to degree 10 (from 0.0648 to 0.0105) and is not better at degree 20 (1.79).
2. Variance is how much the fit moves from one training set to the next, so one fit cannot show it. In practice we keep some data out of the fit and test on it. That is cross-validation, the next section.

</details>


## 3. Cross-validation, and the leakage bug

**Online lab.** Open the [Calibration Lab](https://vsokolov.org/labs/calibration-lab.html) (it uses the same German credit data and the same 1/5 lending cutoff as earlier labs). Try this: switch "Scored on" from the loans it was fitted to to new loans (10-fold CV) and watch the largest model's reliability diagram and profit change.

We have only one data set in real life. Cross-validation (CV) reuses it to estimate the test error.

**10-fold CV, in four steps.**

1. Give each of the 40 rows a fold number from 0 to 9, at random.
2. For each fold $k$: fit the model on the other nine folds, then compute the squared error on fold $k$.
3. The CV error of the model is the average of the 10 fold errors.
4. Do this for every candidate model and compare.

The 10 fold errors also give a standard error: $\mathrm{SE} = \mathrm{sd} / \sqrt{10}$, where sd is their standard deviation. This is the conventional approximate SE used by the one-standard-error rule. The training sets of the folds overlap, so it is a rule of thumb, not an exact uncertainty. The **one-standard-error rule** says: find the model with the lowest CV error, then choose the simplest model whose CV error is within one SE of that lowest value. Here simplest means the smallest degree, and the SE is the one of the model with the lowest CV error. Simpler models are easier to explain and less likely to be overfit.

We use the same simulated training set as Section 1 and the degrees 1 to 12.


In [ ]:
folds = np.random.default_rng(1).permutation(n) % 10     # fold number of each row

def poly_fold_errors(x, y, degree, folds):
    errs = []
    for k in range(10):
        tr = folds != k
        fit = Polynomial.fit(x[tr], y[tr], degree)
        errs.append(np.mean((fit(x[~tr]) - y[~tr]) ** 2))
    return np.array(errs)

degrees = np.arange(1, 13)
fold_err = np.array([poly_fold_errors(x_tr, y_tr, d, folds) for d in degrees])   # shape (12, 10)
cv_mean = fold_err.mean(axis=1)
cv_se = fold_err.std(axis=1, ddof=1) / np.sqrt(10)
best_degree = int(degrees[np.argmin(cv_mean)])
threshold = cv_mean.min() + cv_se[np.argmin(cv_mean)]

print(pd.DataFrame({"CV MSE": cv_mean, "SE": cv_se}, index=degrees).round(4).T)
print("lowest CV error at degree", best_degree, "; one-SE threshold", round(threshold, 4))

plt.figure()
plt.errorbar(degrees, cv_mean, yerr=cv_se, color="C0", marker="o", ms=4, capsize=2, label="10-fold CV MSE")
plt.axhline(threshold, color="C3", ls="--", label="lowest + 1 SE")
plt.yscale("log")
plt.xlabel("polynomial degree")
plt.ylabel("CV MSE (log scale)")
plt.legend()
plt.tight_layout()
plt.show()


**Reading.**

- The lowest CV error is at degree 6, with CV MSE 0.0338 and SE 0.0077. The threshold (lowest plus one SE) is 0.0415.
- The CV error is high at degree 1 (underfitting). It falls to its lowest value at degree 6. It climbs steeply for degrees 9 to 12 (overfitting). It is the same U shape as the test error in Section 1, but computed without any test set.
- Several degrees fall under the threshold. The one-standard-error rule picks the smallest of them. That is the next exercise.


### Exercise 11.1 (In class)

Apply the one-standard-error rule: find the smallest degree whose CV error `cv_mean` is at most `threshold`.

- Inputs: `degrees`, `cv_mean`, `threshold` (all defined above).
- Define: `degree_1se`, one integer.


In [ ]:
# Exercise 11.1: define degree_1se
degree_1se = None


In [ ]:
check("degree_1se", degree_1se, "31c64c60817b098a")


### The planted bug

Below is a cross-validation function for ridge regression with feature screening. Screening means keeping only the 10 features most correlated with the outcome $y$. Ridge is a linear regression with a penalty $\lambda$ on the size of the coefficients. We use it fully in Section 6. Today we only need to know that it fits a model on standardized features.

The function `cv_screen_flawed` has a planted bug. **It leaks information from the validation rows into the fit.** Read it and find the line before you read on.

The data are simulated: 40 rows, 200 features, and an outcome that is pure noise, unrelated to every feature. So no model can predict it. The honest CV error should be about the variance of $y$. We report the CV MSE divided by the variance of $y$. A value below 1 claims skill. A value near or above 1 says there is none.


In [ ]:
def make_noise(seed, n=40, p=200):
    r = np.random.default_rng(seed)
    X = r.standard_normal((n, p))
    y = r.standard_normal(n)
    return X, y, r.permutation(n) % 10

def top_features(X, y, top=10):
    # the 10 features with the largest absolute correlation with y
    corr = np.abs(((X - X.mean(0)) / X.std(0)).T @ (y - y.mean()))
    return np.argsort(corr)[-top:]

def cv_screen_flawed(X, y, folds, lam=5.0):
    keep = top_features(X, y)                       # choose the features
    errs = []
    for k in range(10):
        tr = folds != k
        model = ridge_fit(X[tr][:, keep], y[tr], lam)
        errs.append(np.mean((ridge_predict(model, X[~tr][:, keep]) - y[~tr]) ** 2))
    return np.mean(errs) / y.var()

X_s, y_s, folds_s = make_noise(0)
print("flawed CV MSE / var(y):", round(cv_screen_flawed(X_s, y_s, folds_s), 3))


**Predict.** Which line is the bug? Write down the line, and the operation that has seen the validation labels.

<details><summary>Hint</summary>

`top_features` looks at `y`, and the call uses all 40 rows, validation rows included.

</details>

Now repair it yourself. The reference version comes after the exercise.


### Exercise 11.2 (In class)

Repair the leak. Write a 10-fold CV for the screened ridge in which the 10 features are chosen inside each fold, from the training rows of that fold only. Copy `cv_screen_flawed` and move the choice of features into the loop.

- Inputs: `top_features`, `ridge_fit`, `ridge_predict` (defined above), the data `X_s, y_s, folds_s` (data set 0) and a second data set `X_t, y_t, folds_t = make_noise(7)`. Use `lam = 5.0` and divide the average fold MSE by `y.var()`, as `cv_screen_flawed` does.
- Define: `honest_s` and `honest_t`, the repaired CV MSE divided by the variance of y, on the first and the second data set. Two numbers.


In [ ]:
# Exercise 11.2: define honest_s, honest_t
X_t, y_t, folds_t = make_noise(7)
honest_s = honest_t = None


In [ ]:
check("honest_s", honest_s, "30bfa4d678b14168")
check("honest_t", honest_t, "eeeb0cd67549c9fd")


In [ ]:
def cv_screen(X, y, folds, lam=5.0):
    errs = []
    for k in range(10):
        tr = folds != k
        keep = top_features(X[tr], y[tr])            # features chosen from the training rows only
        model = ridge_fit(X[tr][:, keep], y[tr], lam)
        errs.append(np.mean((ridge_predict(model, X[~tr][:, keep]) - y[~tr]) ** 2))
    return np.mean(errs) / y.var()

flawed_one = cv_screen_flawed(X_s, y_s, folds_s)
honest_one = cv_screen(X_s, y_s, folds_s)
print("flawed CV MSE / var(y):", round(flawed_one, 3))
print("honest CV MSE / var(y):", round(honest_one, 3))

leaky = np.array([cv_screen_flawed(*make_noise(s)) for s in range(50)])
honest = np.array([cv_screen(*make_noise(s)) for s in range(50)])
print("average over 50 simulated data sets")
print("  flawed CV MSE / var(y):", round(leaky.mean(), 3))
print("  honest CV MSE / var(y):", round(honest.mean(), 3))


**Reading.**

- On the first data set, the flawed pipeline reports 0.455 times the variance of $y$ and the honest one 0.876. One data set is one random draw, so the honest value can fall below 1 by chance. The average over 50 data sets is the stable number.
- Over 50 simulated data sets, the flawed pipeline reports 0.595 times the variance of $y$. It claims to explain part of a pure-noise outcome. The honest pipeline reports 1.445. It correctly finds no skill: its value is above 1, because a model fitted to noise predicts worse than the plain average.
- The bug: the 10 features were chosen from all 40 rows, so the validation rows voted on which features to keep. Features that happen to correlate with the validation $y$ values are kept, and the validation error looks good.
- The rule for every step that learns something from data (scaling, picking features, filling in missing values): **learn it on the training rows of the fold, then apply it to the validation rows.**
- The honest version is the fix: the choice sits inside the loop over folds.

**A short note on scaling.** A common version of the same mistake is to standardize all the data (subtract the mean, divide by the standard deviation) before the split. It is still wrong practice. Its effect is usually small, because scaling never looks at $y$. The cell below compares the two versions on 200 simulated data sets (40 rows, 10 features on very different scales, an outcome that depends on three of them).


In [ ]:
def make_data(seed, n=40, p=10):
    r = np.random.default_rng(seed)
    X = r.standard_normal((n, p)) * r.choice([1, 10], p) + r.uniform(-5, 5, p)
    beta = np.zeros(p)
    beta[:3] = np.array([1.0, -1.0, 0.5]) * 3 / X.std(axis=0)[:3]
    y = X @ beta + r.normal(0, 1, n)
    return X, y, r.permutation(n) % 10

def cv_ridge(X, y, lam, folds, leak):
    mu, sd = X.mean(axis=0), X.std(axis=0)          # scaler fitted on ALL rows (used only if leak)
    errs = []
    for k in range(10):
        tr = folds != k
        model = ridge_fit(X[tr], y[tr], lam, mu, sd) if leak else ridge_fit(X[tr], y[tr], lam)
        errs.append(np.mean((ridge_predict(model, X[~tr]) - y[~tr]) ** 2))
    return np.mean(errs)

diffs = np.array([cv_ridge(*make_data(s)[:2], 5.0, make_data(s)[2], True) - cv_ridge(*make_data(s)[:2], 5.0, make_data(s)[2], False)
                  for s in range(200)])
print("scaler fitted before the split: average CV MSE (leaky minus honest):", round(diffs.mean(), 4))
print("share of data sets where the leaky score looked better:", round(np.mean(diffs < 0), 3))


In this simulation the scaler leak changes the CV MSE by 0.0516 on average (leaky minus honest), and the leaky score looked better in 10.0 percent of the data sets. Here the effect is small and in the unflattering direction: the leaky score was a little worse, not better (the noise variance in this simulation is 1). Do it correctly anyway, so you never depend on that luck.

**Check yourself.**

1. A friend picks the 10 best features on the full data, then runs 10-fold CV and reports the CV error. What is wrong, and how large was the effect here?
2. A friend fits a standardizer on the full data, then splits into train and test. How big is the effect likely to be, and should they do it?
3. Why is it enough to choose a degree with CV, and not enough to report the CV error of the chosen degree as the final test error?

<details><summary>Answer</summary>

1. The validation rows took part in choosing the features, so the CV error is flattered. Here the flawed pipeline reported 0.595 times the variance of a pure-noise outcome, the honest one 1.445.
2. Usually small and, here, in the unflattering direction (the leaky score was 0.0516 higher in MSE units on average). They should still fit the standardizer on the training rows only, because the same habit applied to feature selection is large.
3. We looked at the CV errors of 12 models and kept the lowest one. The lowest of many noisy numbers is flattered. A final honest number needs data that took no part in choosing the model.

</details>


### Computing corner: standardize with training statistics only

Two reasons to standardize before a penalty.

**Reason 1: the penalty depends on the units.** The same ridge penalty $\lambda \sum_j \beta_j^2$ costs different amounts for the same effect, depending on the unit of the feature. Take the German credit data. We predict the 0/1 default flag from the borrower's age (in years) and the loan amount. We fit the same ridge with penalty $\lambda = 10^5$ twice: once with the amount in dollars and once in thousands of dollars. The data are the same. Only the unit changed. To compare fairly, the table shows the effect of a one-standard-deviation change in each feature.


In [ ]:
credit = pd.read_csv(DATA / "credit.csv")
y_c = credit["Default"].to_numpy(float)
age = credit["age"].to_numpy(float)
amount = credit["amount"].to_numpy(float)

def ridge_raw(X, y, lam):
    # no standardizing: center only, then penalize the coefficients as they are
    Xc = X - X.mean(axis=0)
    return np.linalg.solve(Xc.T @ Xc + lam * np.eye(X.shape[1]), Xc.T @ (y - y.mean()))

lam = 1e5
rows = []
for label, scale in (("amount in dollars", 1.0), ("amount in thousands", 1000.0)):
    X = np.column_stack([age, amount / scale])
    b = ridge_raw(X, y_c, lam)
    b_free = ridge_raw(X, y_c, 0.0)
    rows.append((label, *(b * X.std(axis=0)), *(b_free * X.std(axis=0))))
units = pd.DataFrame(rows, columns=["units", "age (ridge)", "amount (ridge)", "age (no penalty)", "amount (no penalty)"]).set_index("units")
print(units.round(4))


**Reading.**

- With no penalty the two rows agree (the last two columns): changing the unit does not change a plain regression.
- With the penalty they disagree. With the amount in dollars, the ridge effect of the amount is 0.0717 per standard deviation, close to the unpenalized 0.0724: in dollars, a tiny coefficient already gives a big effect, and its penalty costs almost nothing. With the amount in thousands, the same effect needs a coefficient 1000 times larger, which costs a much larger penalty, so the effect is shrunk to 0.0053.
- The age effect also changes between the two rows (-0.0249 against -0.0236), because the two coefficients share one penalty.
- A penalty that depends on the unit of measure is not a modelling choice. Standardizing removes the unit.

**Reason 2: where the numbers come from.** The standardizer is part of the model. It has two numbers per feature, the mean and the standard deviation. They must come from the training rows only, and the same two numbers are then applied to validation and test rows. In code:

```python
mu, sd = X_train.mean(axis=0), X_train.std(axis=0)
Z_train = (X_train - mu) / sd
Z_test = (X_test - mu) / sd      # the training mu and sd, not new ones
```

The test set then has mean near, but not exactly, zero. That is correct. A model in production sees one new row at a time and cannot compute a mean from it, so the training numbers are the only ones available.


## 4. Choosing by evidence: the death-penalty data

Cross-validation chooses a model by how well it predicts held-out data. Bayes chooses by **evidence**: how likely the data are under the model, averaged over the model's prior. We use a table about death sentences, which we met when we learned a rate with a Beta prior.

The data are real (Radelet, 1981): 326 defendants charged with murder in 20 Florida counties, 1976 to 1977. A **case** is one defendant. The **rate** is the share of cases that ended in a death sentence. A **cell** is one combination of the defendant's race and the victim's race. For example, in the cell white defendant, white victim, there were 19 death sentences in 151 cases. Use only the labels in the table below.


In [ ]:
cell_names = ["white defendant, white victim", "white defendant, Black victim",
              "Black defendant, white victim", "Black defendant, Black victim"]
deaths = np.array([19, 0, 11, 6])        # death sentences
n_cases = np.array([151, 9, 63, 103])    # cases
table = pd.DataFrame({"cases": n_cases, "death sentences": deaths,
                      "rate": deaths / n_cases}, index=cell_names)
print(table.round(3))
print("overall rate:", round(deaths.sum() / n_cases.sum(), 3))


**Four models for the death-penalty rate.** In every model the rate in a group is an unknown probability $\theta$, with a uniform Beta(1, 1) prior (the Beta prior for a rate that we used earlier).

- M1, one rate: one $\theta$ for everyone.
- M2, by victim: one $\theta$ for white victims and one for Black victims.
- M3, by defendant: one $\theta$ for white defendants and one for Black defendants.
- M4, four cells: a separate $\theta$ for each of the four cells.

**The evidence of a model** is the probability of the data, averaged over the prior. For a group with $k$ death sentences out of $m$ cases and a Beta(1, 1) prior on $\theta$, the evidence of the sequence of outcomes is the Beta function

$$
\int_0^1 \theta^k (1-\theta)^{m-k}\, d\theta = B(k+1,\; m-k+1).
$$

A model with several groups multiplies one such factor per group. We work with logs, so products become sums, and `scipy.special.betaln` gives $\log B$ without overflow.

Equal prior weight on the four models gives posterior model probabilities $P(M_j \mid \text{data}) = E_j / \sum_i E_i$, where $E_j$ is the evidence of model $j$.


In [ ]:
def log_evidence(k, m, a=1.0, b=1.0):
    # log evidence of one group: k successes out of m, Beta(a, b) prior on the rate
    return betaln(a + k, b + m - k) - betaln(a, b)

def model_groups(deaths, n_cases):
    # the cells are ordered: WW, WB, BW, BB (defendant, victim)
    d, m = np.asarray(deaths), np.asarray(n_cases)
    return {
        "M1 one rate": [(d.sum(), m.sum())],
        "M2 by victim": [(d[0] + d[2], m[0] + m[2]), (d[1] + d[3], m[1] + m[3])],
        "M3 by defendant": [(d[0] + d[1], m[0] + m[1]), (d[2] + d[3], m[2] + m[3])],
        "M4 four cells": list(zip(d, m)),
    }

def model_probs(deaths, n_cases, a=1.0, b=1.0):
    groups = model_groups(deaths, n_cases)
    logE = np.array([sum(log_evidence(k, m, a, b) for k, m in g) for g in groups.values()])
    P = np.exp(logE - logE.max())
    return pd.DataFrame({"evidence": np.exp(logE), "probability": P / P.sum()}, index=list(groups))

post = model_probs(deaths, n_cases)
pd.set_option("display.float_format", lambda v: f"{v:.3g}")
print(post)
pd.reset_option("display.float_format")


**Reading.**

- The evidences are tiny numbers (between $10^{-52}$ and $10^{-51}$), because each is the probability of one exact sequence of 326 outcomes. Only their ratios matter.
- The posterior probabilities are 0.354 for one rate, 0.590 for by victim, 0.034 for by defendant and 0.022 for four cells.
- In these data, the model that uses only the victim's race is best supported, with about 59% of the posterior probability. Death sentences were 14.0% of cases with white victims and 5.4% of cases with Black victims. The defendant's race model (0.034) is behind even the one-rate model (0.354).
- Within the white-victim cases, Black defendants had the higher rate: 11 of 63 (0.175), against 19 of 151 (0.126) for white defendants.
- This describes an association in 326 old cases. It does not show why the sentences differed.
- These probabilities are conditional on the four candidate models, their Beta(1, 1) priors, the equal prior weights, and the binomial sampling model. They do not settle the social question. A different prior would give different probabilities.
- The four-cell model fits the counts best (each cell gets its own observed rate), yet has the smallest posterior probability.

**BIC in one line.** For large samples the log evidence is close to $-\tfrac12 \mathrm{BIC}$, where $\mathrm{BIC} = q \ln N - 2 \ln \hat L$, $q$ is the number of parameters, $N$ the number of cases and $\hat L$ the best likelihood. It charges $\ln N$ per parameter. It is the large-sample shortcut to the evidence, an asymptotic approximation under regularity conditions. One cell here has 0 death sentences in 9 cases, so we keep the exact calculation and treat the BIC numbers only as a rough second opinion.


In [ ]:
def max_loglik(groups):
    ll = 0.0
    for k, m in groups:
        th = k / m
        ll += (k * np.log(th) if k > 0 else 0.0) + ((m - k) * np.log(1 - th) if m - k > 0 else 0.0)
    return ll

bic = pd.Series({name: len(g) * np.log(n_cases.sum()) - 2 * max_loglik(g)
                 for name, g in model_groups(deaths, n_cases).items()})
print(bic.round(1))


BIC prefers the smallest value. It ranks the four models in the same order as the posterior probabilities: the lowest BIC is M2 by victim at 231.8.

**Check yourself.** The four-cell model fits the counts best. Why does it get the smallest posterior probability?

<details><summary>Answer</summary>

Its prior spreads over four rates, and most of the prior mass lies on rates the data rule out (for example, the cell of 9 cases has no death sentences, yet a uniform prior gives a rate near 0.5 in that cell as much weight as a rate near 0). The evidence is the prior-averaged fit, so it charges for flexibility. It is the same lesson as the degree 20 polynomial in Section 1: the best fit is not the best model.

</details>

**Your turn.** Double every count (twice the cases and twice the death sentences, same rates) and recompute the four probabilities. This is a hypothetical new independent data set with the same proportions. Copying the existing rows twice would not create new evidence. Target: say which model leads now, and give its probability.


In [ ]:
scale = 1          # change this to 2
print(model_probs(deaths * scale, n_cases * scale).probability.round(3))


In [ ]:
post2 = model_probs(deaths * 2, n_cases * 2).probability
print("doubled counts:")
print(post2.round(3))


With twice the data and the same rates, the winner stays the same (by victim) and its probability goes up from 0.590 to 0.929. The one-rate model falls from 0.354 to 0.035. More data make the evidence more decisive, as they should.


### Exercise 11.3 (In class)

Compute the posterior model probabilities with equal prior weight and Beta(1, 1) priors, using `scipy.special.betaln`. Write the log evidence of each model yourself. Do not call `model_probs`.

- Inputs: `deaths`, `n_cases` (arrays of 4, order: white defendant white victim, white defendant Black victim, Black defendant white victim, Black defendant Black victim).
- Define: `p_one`, `p_victim`, `p_defendant`, `p_cells`: four numbers that add to 1.


In [ ]:
# Exercise 11.3: define p_one, p_victim, p_defendant, p_cells
p_one = p_victim = p_defendant = p_cells = None


In [ ]:
check("p_one", p_one, "e954c44fee01ba2a")
check("p_victim", p_victim, "493b4751cc1c99b0")
check("p_defendant", p_defendant, "b3111747706834dc")
check("p_cells", p_cells, "7bbbf58ba2727421")


## 5. James-Stein: learning the prior from the data

Here is a real estimation problem. In the 1970 baseball season, Efron and Morris took 18 players and looked at their batting average after the first 45 at-bats. Then they looked at the rest of the season. The question: what is the best guess of each player's true skill?

- The obvious estimate is each player's own early average. 45 at-bats is little data, so these averages are noisy, and the extreme ones are partly luck.
- The **James-Stein** estimate pulls every average toward the average of all 18 players. The pull is the same fraction for every player.

The estimate, in three steps.

1. Let $\bar y$ be the average of the 18 early averages, and $y_i$ the early average of player $i$.
2. Estimate the noise variance of one early average as $s^2 = \bar y (1 - \bar y)/45$. That is the variance of a proportion from 45 at-bats.
3. The shrinkage factor is $c = 1 - \dfrac{(K - 3)\, s^2}{\sum_i (y_i - \bar y)^2}$, where $K = 18$ is the number of players. The estimate for player $i$ is $\hat y_i = \bar y + c\,(y_i - \bar y)$.

This is an approximate James-Stein-style rule for proportions. It is not a direct use of the exact theorem, which is for Gaussian data with a known variance. The rest-of-season average is a noisy check on skill, not skill itself.

This is **empirical Bayes**. A Bayesian puts a prior on the players' skills: they come from a bell curve centred at some value. Here the centre and the spread of that prior are estimated from the data themselves, from all 18 players. The shrinkage factor $c$ is the weight on the player's own average, and $1 - c$ is the weight on the grand mean. When the players are close to each other relative to the noise, $c$ is small and the pull toward the grand mean is strong.


In [ ]:
bb = pd.read_csv(DATA / "EfronMorrisBB.txt", sep="\t")
bb["name"] = bb.FirstName + " " + bb.LastName
y_early = bb.BattingAverage.to_numpy()          # first 45 at-bats
y_rest = bb.RemainingAverage.to_numpy()         # rest of the season (the truth we will judge by)
K = len(bb)

y_bar = y_early.mean()
s2 = y_bar * (1 - y_bar) / 45
c = 1 - (K - 3) * s2 / np.sum((y_early - y_bar) ** 2)
y_js = y_bar + c * (y_early - y_bar)

print("players:", K, "  grand mean:", round(y_bar, 4), "  shrinkage factor c:", round(c, 4))
print("mean squared error against the rest of the season (mean over the 18 players)")
print("  raw early averages:", round(np.mean((y_early - y_rest) ** 2), 5))
print("  James-Stein       :", round(np.mean((y_js - y_rest) ** 2), 5))

plt.figure(figsize=(5, 3.5))
plt.scatter(y_rest, y_early, color="C0", marker="o", s=18, label="early average")
plt.scatter(y_rest, y_js, color="C3", marker="x", s=22, label="James-Stein")
plt.plot([0.18, 0.40], [0.18, 0.40], color="black", ls="--", lw=1, label="perfect")
plt.xlabel("average over the rest of the season")
plt.ylabel("estimate")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

shrink = pd.DataFrame({"early": y_early, "James-Stein": y_js, "rest": y_rest, "moved": y_js - y_early}, index=bb.name)
print(shrink.reindex(shrink.moved.abs().sort_values(ascending=False).index).head(4).round(3))


**Reading.**

- The shrinkage factor is $c =$ 0.212. Every early average is pulled 79% of the way to the grand mean 0.265.
- Against the rest of the season, the raw averages have a mean squared error of 0.00476 and the James-Stein estimates 0.00149. That is about 3.2 times smaller. Exercise 11.4 asks for these two means.
- In the plot, the blue circles (early averages) range from 0.156 to 0.400. The rest-of-season averages (horizontal axis) range only from 0.200 to 0.346. The red crosses (James-Stein) range from 0.242 to 0.294, much closer to the truth.
- The player who moved most is Roberto Clemente, from 0.400 to 0.294. His rest-of-season average was 0.346.

**Check yourself.**

1. Which players move the most, and why is that reasonable? What is the limit of the method?
2. In what sense is the prior "learned from the data"?

<details><summary>Answer</summary>

1. The players with the most extreme early averages are shrunk most in absolute terms, because the move is $(1-c)(y_i - \bar y)$. Every player gets the same fraction, but that alone does not make it fair: the rule lowers the total error on average, and it can still move a single player the wrong way. An extreme average after 45 at-bats is partly skill and partly luck, and the luck does not repeat. The shrinkage takes out the average amount of luck.
2. The centre ($\bar y$) and the amount of shrinkage ($c$) are computed from the 18 averages. We did not choose a prior by hand. This is empirical Bayes.

</details>


### Exercise 11.4 (In class)

Compute the James-Stein shrinkage factor and compare its error with the raw early averages.

- Inputs: `y_early` (18 early averages), `y_rest` (18 rest-of-season averages), `K = 18`, and 45 at-bats per early average. Use the three steps above (grand mean, $s^2 = \bar y (1-\bar y)/45$, then $c$).
- Define: `js_factor` (the factor $c$), `err_raw` (the mean over players of $(y_i - \text{rest}_i)^2$ for the raw averages), `err_js` (the same mean for the James-Stein estimates). Three numbers.


In [ ]:
# Exercise 11.4: define js_factor, err_raw, err_js
js_factor = err_raw = err_js = None


In [ ]:
check("js_factor", js_factor, "69634ff0e83dc474")
check("err_raw", err_raw, "d8606c8ca66b7daf")
check("err_js", err_js, "d4aba3b20e9bcf1c")


## 6. A prior is a regularizer: ridge and lasso

Linear regression with many features overfits for the same reason as the degree 20 polynomial. The cure is to add a penalty on the coefficients.

**Ridge** minimizes

$$
\sum_i (y_i - x_i^\top \beta)^2 + \lambda \sum_j \beta_j^2.
$$

**Lasso** minimizes the same squared error plus $\lambda \sum_j |\beta_j|$.

$\lambda \ge 0$ sets the strength of the penalty. At $\lambda = 0$ we get ordinary least squares.

**Online lab.** Open [Needle in a Haystack](https://vsokolov.org/labs/sparsity-lab.html) next to this section. Try this: fit least squares, ridge and the lasso on the 1,000 genes and 100 patients, and count how many genes each method keeps. Only the lasso sets coefficients to exactly zero.

**Where the penalty comes from: a prior.** Suppose $y_i = x_i^\top \beta + \varepsilon_i$ with noise $\varepsilon_i \sim N(0, \sigma^2)$, and suppose we believe before seeing data that each coefficient is small. The MAP estimate (maximum a posteriori, the most probable $\beta$ after seeing the data) maximizes the posterior, which means minimizing the negative log of likelihood times prior.

- A Gaussian prior $\beta_j \sim N(0, \tau^2)$ gives the penalty $\lambda \sum_j \beta_j^2$, with $\lambda = \sigma^2 / \tau^2$. **Ridge is MAP with a Gaussian prior.** A tighter prior (small $\tau$) is a bigger $\lambda$.
- A Laplace prior (a sharp peak at 0 and heavy tails) gives the penalty $\lambda \sum_j |\beta_j|$. **Lasso is MAP with a Laplace prior.**
- In deep learning the ridge penalty is called **weight decay**: each gradient step first shrinks every weight by a small fraction, $\beta \leftarrow (1 - 2\eta\lambda)\beta - \eta \cdot \text{gradient of the squared error}$, where $\eta$ is the step size. The factor 2 comes from differentiating $\lambda\beta^2$. It is the same Gaussian prior. Careful with libraries: PyTorch's SGD option `weight_decay` $= w$ adds $w\beta$ to the gradient, which is the gradient of $\tfrac{w}{2}\lVert\beta\rVert^2$, so it matches our $\lambda$ when $w = 2\lambda$. Deep learning code also often applies weight decay to every parameter, biases included, unless the parameters are split into groups. Our `ridge_fit` leaves the intercept unpenalized.

**The data are simulated:** 60 rows, 20 standard normal features, and an outcome that depends on only the first 5 features (true coefficients 3, -2, 1.5, 1, -1), with noise sd 3. The other 15 features are noise. The coefficient paths below use the features standardized once on all 60 rows. That is only to draw the picture, not to score a model. Every validation score in this section scales inside the folds. Because the features are already on one scale, standardizing changes little here, but our helper `ridge_fit` does it anyway. It penalizes the coefficients of the standardized features only. The intercept is not penalized: `ridge_fit` centers $y$ and the features first.

A warning about the symbol: in this notebook $\lambda$ multiplies the penalty, and the squared error is a *sum* over the rows. If the squared error is a *mean* over $N$ rows instead (as in the Exercise 11.6 setup at the end), the same fit needs $\lambda_{\text{mean}} = \lambda_{\text{sum}} / N$. scikit-learn's `Lasso` minimizes $\frac{1}{2n}\sum_i (y_i - x_i^\top\beta)^2 + \alpha \sum_j |\beta_j|$, with $n$ the number of rows, so its `alpha` is on another scale. The ridge grid `lams` and the lasso grid `alphas` are not comparable. Only the shape of the paths is.


In [ ]:
from sklearn.linear_model import Lasso, lasso_path

def make_sparse(k_true, seed=6, n=60, p=20):
    r = np.random.default_rng(seed)
    X = r.standard_normal((n, p))
    beta = np.zeros(p)
    beta[:k_true] = np.array([3, -2, 1.5, 1, -1, 2, -1.5, 1, 2, -2])[:k_true]
    y = X @ beta + r.normal(0, 3, n)
    return X, y, beta

X_r, y_r, beta_true = make_sparse(5)
folds_r = np.random.default_rng(2).permutation(len(y_r)) % 10
lams = np.logspace(-1, 3, 25)

# coefficient paths (all on the standardized scale)
ridge_path = np.array([ridge_fit(X_r, y_r, l)[-1] for l in lams])
Z_r = (X_r - X_r.mean(axis=0)) / X_r.std(axis=0)
alphas = np.logspace(-2, 1, 40)
alphas_desc, lasso_coefs, _ = lasso_path(Z_r, y_r - y_r.mean(), alphas=alphas)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), sharey=True)
for j in range(20):
    sty = dict(color="C0", ls="-", lw=1.5) if beta_true[j] != 0 else dict(color="gray", ls="--", lw=0.8)
    axes[0].plot(lams, ridge_path[:, j], **sty)
    axes[1].plot(alphas_desc, lasso_coefs[j], **sty)
axes[0].set_xscale("log"); axes[1].set_xscale("log")
axes[0].set_title("ridge", fontsize=9); axes[1].set_title("lasso", fontsize=9)
axes[0].set_xlabel("penalty strength (lambda)"); axes[1].set_xlabel("penalty strength (lasso alpha)")
axes[0].set_ylabel("coefficient (standardized features)")
axes[0].plot([], [], color="C0", ls="-", label="truly nonzero"); axes[0].plot([], [], color="gray", ls="--", label="truly zero")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

print("ridge: coefficients exactly zero at the largest lambda:", int(np.sum(ridge_path[-1] == 0)))
print("lasso: coefficients exactly zero at the largest alpha:", int(np.sum(lasso_coefs[:, 0] == 0)))
# the alpha at which each coefficient is last nonzero (alphas is increasing, lasso_path returns them decreasing)
alpha_dies = np.array([alphas_desc[np.nonzero(lasso_coefs[j])[0]].max() if np.any(lasso_coefs[j]) else 0.0 for j in range(20)])
print("typical alpha where a coefficient reaches 0, truly nonzero:", round(float(np.median(alpha_dies[:5])), 3),
      " truly zero:", round(float(np.median(alpha_dies[5:])), 3))


**Reading.**

- Both paths start at the least squares values on the left (small penalty) and shrink as the penalty grows.
- Ridge (left) shrinks all 20 coefficients smoothly toward 0 and never reaches it: at the largest $\lambda$ the number of exact zeros is 0 of 20.
- Lasso (right) sets coefficients to exactly 0 one after another: at the largest alpha the number of exact zeros is 20 of 20. The truly nonzero coefficients survive to a larger penalty: the median alpha at which they reach 0 is 1.70, against 0.20 for the truly zero ones.
- The sharp peak of the Laplace prior at 0 is what produces exact zeros. The smooth Gaussian prior only pulls.

**Choosing the penalty by CV.** Each $\lambda$ on the grid `lams` is scored by 10-fold CV with the fold numbers `folds_r`. The scaler is fitted inside each fold, as in Section 3.


In [ ]:
def cv_ridge_lambda(X, y, lam, folds):
    errs = []
    for k in range(10):
        tr = folds != k
        model = ridge_fit(X[tr], y[tr], lam)
        errs.append(np.mean((ridge_predict(model, X[~tr]) - y[~tr]) ** 2))
    return np.mean(errs)

cv_demo = np.array([cv_ridge_lambda(X_r, y_r, l, folds_r) for l in lams])
print("CV MSE at lambda =", round(lams[0], 2), ":", round(cv_demo[0], 2))
print("CV MSE at lambda =", round(lams[-1], 0), ":", round(cv_demo[-1], 2))
print("lowest CV MSE:", round(cv_demo.min(), 3), " at lambda =", round(lams[int(np.argmin(cv_demo))], 2))


In [ ]:
def lasso_cv_fit(X, y, folds, alphas):
    # choose the lasso alpha by 10-fold CV; the scaler is fitted inside each fold, on the training rows only
    scores = []
    for a in alphas:
        errs = []
        for k in range(10):
            tr = folds != k
            mu, sd = X[tr].mean(axis=0), X[tr].std(axis=0)
            m = Lasso(alpha=a, max_iter=20000).fit((X[tr] - mu) / sd, y[tr])
            errs.append(np.mean((m.predict((X[~tr] - mu) / sd) - y[~tr]) ** 2))
        scores.append(np.mean(errs))
    best = alphas[int(np.argmin(scores))]
    mu, sd = X.mean(axis=0), X.std(axis=0)           # final model: all training rows
    final = Lasso(alpha=best, max_iter=20000).fit((X - mu) / sd, y)
    return best, final.coef_

alpha_cv, lasso_coef = lasso_cv_fit(X_r, y_r, folds_r, alphas)      # the same folds as ridge
lam_cv = lams[int(np.argmin(cv_demo))]
ridge_coef = ridge_fit(X_r, y_r, lam_cv)[-1]
true_idx = np.arange(20) < 5
print("ridge at the CV lambda: nonzero coefficients:", int(np.sum(ridge_coef != 0)))
print("lasso at the CV alpha :", round(alpha_cv, 3), " nonzero coefficients:", int(np.sum(lasso_coef != 0)),
      "(of which truly nonzero:", int(np.sum((lasso_coef != 0) & true_idx)), ")")
print("size of the 15 truly-zero coefficients (average absolute value):")
print("  ridge:", round(np.mean(np.abs(ridge_coef[~true_idx])), 3), "  lasso:", round(np.mean(np.abs(lasso_coef[~true_idx])), 3))


**Reading.**

- The CV choice of ridge penalty is $\lambda =$ 14.68. At that value ridge still has 20 nonzero coefficients out of 20: it shrinks, it does not select.
- Lasso at its CV choice (alpha 0.346, on its own scale, with the same folds as ridge) keeps 10 coefficients: 5 of the 5 truly nonzero ones plus 5 noise features. The scaler was fitted inside each fold, as in Section 3.
- The 15 truly-zero coefficients are smaller on average under lasso (0.091) than under ridge (0.288).

**Your turn.** The cell below counts the nonzero lasso coefficients for a different number of truly nonzero coefficients. It runs as given with `k_true = 5`. Change it to 2, then to 10. Target: describe how the number of nonzero lasso coefficients moves with `k_true`, and whether lasso still finds the true ones.


In [ ]:
k_true = 5            # change this to 2, then 10
Xk, yk, bk = make_sparse(k_true)
a_k, coef_k = lasso_cv_fit(Xk, yk, folds_r, alphas)
print("true nonzero:", k_true, "  lasso kept:", int(np.sum(coef_k != 0)),
      "  of which truly nonzero:", int(np.sum((coef_k != 0) & (bk != 0))))


**Check yourself.**

1. If a prior says "coefficients are probably small", is ridge a model of the data, or a belief about the coefficients?
2. Lasso set coefficients to exactly zero and ridge did not. Which would you use if you want a short list of candidate features, and how far should you trust the list?

<details><summary>Answer</summary>

1. A belief about the coefficients, written as a prior. The penalty is the negative log of that prior.
2. Lasso gives a short list of candidate features, because some coefficients are exactly zero. It is not a definitive list of the features that matter: in this example it kept 5 irrelevant features, and the list can change with another sample. Judge it by how well the model predicts held-out data, not as a scientific conclusion.

</details>

### Computing corner: solve, do not invert

Ridge has a closed form. Set the gradient of the penalized squared error to zero and get a linear system

$$
(X^\top X + \lambda I)\,\beta = X^\top y.
$$

The formula in many textbooks is $\beta = (X^\top X + \lambda I)^{-1} X^\top y$. In code there are two ways to compute it: `np.linalg.inv(A) @ b` (build the inverse, then multiply) and `np.linalg.solve(A, b)` (solve the system directly). Both give the same answer on a clean problem. On a nearly collinear $X$ (two features almost the same) they do not.

The **condition number** of a matrix measures how much it magnifies small errors. A condition number of $10^{k}$ means you lose about $k$ of the 16 digits a decimal number carries. We build a simulated $X$ with 2000 rows and 50 features, where the second feature equals the first plus noise of size $10^{-6}$.


In [ ]:
import time

rng_c = np.random.default_rng(0)
Xn = rng_c.standard_normal((2000, 50))
Xn[:, 1] = Xn[:, 0] + 1e-6 * rng_c.standard_normal(2000)       # nearly collinear
beta_star = np.zeros(50); beta_star[2:7] = 1.0                 # the truth does not use features 0 and 1
yn = Xn @ beta_star + 0.1 * rng_c.standard_normal(2000)

G, rhs = Xn.T @ Xn, Xn.T @ yn
rows = []
for lam_c in (0.0, 1.0):
    A = G + lam_c * np.eye(50)
    t0 = time.perf_counter(); b_inv = np.linalg.inv(A) @ rhs; t_inv = time.perf_counter() - t0
    t0 = time.perf_counter(); b_sol = np.linalg.solve(A, rhs); t_sol = time.perf_counter() - t0
    rows.append((lam_c, np.linalg.cond(A),
                 np.linalg.norm(A @ b_inv - rhs), np.linalg.norm(A @ b_sol - rhs),
                 np.linalg.norm(b_inv - beta_star), np.linalg.norm(b_sol - beta_star), t_inv, t_sol))
solve_tbl = pd.DataFrame(rows, columns=["lambda", "cond number", "residual inv", "residual solve",
                                        "coef error inv", "coef error solve", "time inv (s)", "time solve (s)"]).set_index("lambda")
pd.set_option("display.float_format", lambda v: f"{v:.3g}")
print(solve_tbl.T)
pd.reset_option("display.float_format")


**Reading.**

- With $\lambda = 0$ the condition number of $X^\top X$ is 4.0e+12. About 12 or more of the 16 digits are gone. The residual $\|A\beta - X^\top y\|$ (how badly the answer fails the equation) is 3.0e-02 for `inv` and 1.4e-09 for `solve`. Solve fails the equation by far less.
- With $\lambda = 1$ the condition number falls to 3.9e+03. Adding $\lambda I$ lifts the tiny eigenvalues of $X^\top X$ away from zero. This is a second job of the ridge penalty: it makes the problem solvable.
- The coefficient error against the true $\beta$ is 2.1e+03 (inv) and 2.1e+03 (solve) at $\lambda = 0$, and 0.01 and 0.01 at $\lambda = 1$. Without the penalty both fail to recover $\beta$ (the same large error), because two features carry the same information and the fit splits it between them with huge opposite coefficients. No way of solving the system can fix that. The penalty ties the two coefficients together.
- The two timing rows are printed in the table (they change a little on every run and on every machine). `solve` is faster, because it never builds the full inverse.
- Rule: `solve` leaves a much smaller residual and is faster, so it is still the right habit: never write `inv(A) @ b`. Use `solve(A, b)`.


## 7. Loss as negative log-likelihood

Every loss function we have used is the negative log of a probability model for the outcome. We fit by choosing the parameters that make the observed data most likely, and the likelihood of independent data is a product, so its log is a sum. Minimizing the **negative log-likelihood** (NLL) is the same as maximizing the likelihood.

| Outcome | Probability model | NLL per observation |
|---|---|---|
| Number, with noise | Gaussian, mean $\hat y$, sd 1 | $\tfrac12 (y - \hat y)^2 + \text{constant}$: squared error |
| Yes or no ($y = 1$ or $0$) | Bernoulli, $P(y=1) = p$ | $-y \log p - (1-y)\log(1-p)$: binary log loss |
| One of $K$ classes | Categorical, probabilities $p_1, \dots, p_K$ | $-\sum_k y_k \log p_k$: softmax cross-entropy |

**The $K$-class case in full.**

1. The class is coded as a one-hot vector $y = (y_1, \dots, y_K)$: all zeros, with a 1 in the place of the true class. In the code, `true_class` and `yb` hold the index of the true class instead.
2. The model produces $K$ scores $z = (z_1, \dots, z_K)$, any real numbers.
3. The softmax turns scores into probabilities: $p_k = e^{z_k} / \sum_j e^{z_j}$. In exact arithmetic they are positive and add to 1. (On a computer a very small probability can round to exactly 0. That is why the code below works with the log of the softmax directly.)
4. The loss is $-\sum_k y_k \log p_k$. Since only one $y_k$ is 1, this is just **$-\log p$ of the true class**.
5. For $K = 2$ it reduces to the binary log loss: $p_1 = 1/(1 + e^{-(z_1 - z_2)})$, which is the logistic function of the score difference.

**By hand, three classes.** Scores $z = (2.0, 1.0, 0.1)$ and the true class is the first one ($y = (1, 0, 0)$). First $e^{2.0} = 7.389$, $e^{1.0} = 2.718$, $e^{0.1} = 1.105$. Their sum is 11.212. The probabilities are $p = (0.659, 0.242, 0.099)$ and the loss is $-\log 0.659 = 0.417$. The code checks it.


In [ ]:
def log_softmax(z):
    # log of the softmax, computed with the log-sum-exp trick: log p_k = z_k - log(sum_j exp(z_j))
    m = z.max(axis=-1, keepdims=True)           # subtracting the max avoids overflow and changes nothing
    return z - m - np.log(np.exp(z - m).sum(axis=-1, keepdims=True))

def softmax(z):
    return np.exp(log_softmax(z))

def cross_entropy(Z, idx):
    # average over the rows of Z of -log p of the true class, computed from the scores (logits)
    Z = np.atleast_2d(Z)
    return float(np.mean(-log_softmax(Z)[np.arange(len(Z)), idx]))

z3 = np.array([2.0, 1.0, 0.1])
y3 = np.array([1, 0, 0])
p3 = softmax(z3)
loss3 = -np.sum(y3 * log_softmax(z3))
print("probabilities:", p3.round(3), " sum:", p3.sum().round(3))
print("loss:", round(loss3, 3), "  = -log p of the true class:", round(-np.log(p3[0]), 3), "  cross_entropy():", round(cross_entropy(z3, [0]), 3))

# why we compute it from the scores: the log of a softmax breaks for very different scores
z_big = np.array([0.0, 1000.0])                  # the true class is the first one
p_first = np.exp(z_big - z_big.max())[0] / np.exp(z_big - z_big.max()).sum()
with np.errstate(divide="ignore"):
    naive = -np.log(p_first)
print("true class 0 with scores", z_big, ": log(softmax) gives", naive, "  log-sum-exp gives", cross_entropy(z_big, [0]))

# the K = 2 case equals the binary log loss
z2 = np.array([0.8, -0.4])
p_yes = 1 / (1 + np.exp(-(z2[0] - z2[1])))
print("K=2 cross-entropy:", round(cross_entropy(z2, [0]), 6), "  binary log loss:", round(-np.log(p_yes), 6))

# Gaussian: NLL against half the sum of squares
yy, mm = np.array([1.0, 2.0, 4.0]), np.array([1.5, 2.0, 3.0])
nll_gauss = np.sum(0.5 * np.log(2 * np.pi) + 0.5 * (yy - mm) ** 2)
print("Gaussian NLL minus constant:", round(nll_gauss - len(yy) * 0.5 * np.log(2 * np.pi), 3), "  half the sum of squares:", round(0.5 * np.sum((yy - mm) ** 2), 3))


**Reading.**

- The three probabilities are 0.659, 0.242, 0.099, and they add to 1. The loss is 0.417, which equals $-\log$ of the first probability. It matches the hand calculation.
- With scores (0, 1000) the first probability rounds to exactly 0 on a computer, so taking the log of the softmax gives infinity. The log-sum-exp form gives the correct loss, 1000. Compute cross-entropy from the scores, not from `log(softmax(...))`.
- For $K = 2$, cross-entropy and the binary log loss are the same number, 0.263282.
- The Gaussian NLL, without its constant, is half the sum of squares. So least squares is maximum likelihood with Gaussian noise.

**Your turn.** The skeleton below averages the loss over a batch of 5 examples with 3 classes, but it loops over only the first example. Change `n_use` to 5 and run it. Target: an average loss of 0.4626. (`true_class` holds the index of the true class of each example.)


In [ ]:
Z5 = np.array([[2.0, 1.0, 0.1],
               [0.5, 2.5, 0.3],
               [1.0, 1.2, 1.1],
               [0.2, 0.1, 3.0],
               [1.5, 0.5, 0.0]])
true_class = np.array([0, 1, 2, 2, 0])

n_use = 1          # change this to 5
losses = [cross_entropy(Z5[i], [true_class[i]]) for i in range(n_use)]
print("average loss over", n_use, "example(s):", round(np.mean(losses), 4))


**Predict.** Before the next cell: the true class has probability 0.9, 0.5, 0.1, 0.01 or 0.001. Will the loss for 0.01 be about 10 times, 40 times, or 400 times the loss for 0.9?


In [ ]:
p_true = np.array([0.9, 0.5, 0.1, 0.01, 0.001])
print(pd.DataFrame({"p of true class": p_true, "loss = -log p": -np.log(p_true)}).round(3).to_string(index=False))


**Reading.** The loss is 0.105 at $p = 0.9$, 0.693 at $p = 0.5$, 2.303 at $p = 0.1$ and 4.605 at $p = 0.01$. The loss at 0.01 is about 44 times the loss at 0.9. It grows without bound as $p \to 0$.

**Check yourself.**

1. Why does a confident wrong prediction cost so much more than a hesitant one?
2. A classifier gives the true class probability 0 on one example. What is the loss, and what does that say about training?

<details><summary>Answer</summary>

1. The loss is $-\log p$ of the true class. A confident wrong prediction gives the true class a very small $p$, and $-\log p$ is large there. A hesitant one gives it moderate $p$. The log makes the penalty steep near 0, which pushes the model to never be sure of something that can happen.
2. The loss is infinite. In exact arithmetic a softmax probability is never exactly 0, but a computer can round a very small one to 0. Computing the loss from the scores with log-sum-exp avoids taking the log of that zero. For one example, the gradient of the loss with respect to score $k$ is $p_k - y_k$, so it never exceeds 1 in size, even when the loss is huge. A confidently wrong example has a huge loss and the largest gradient. A few such examples can dominate training.

</details>


In [ ]:
Zb = np.array([[1.2, 0.3, -0.5, 0.0],
               [0.1, 2.2, 0.4, -1.0],
               [-0.3, 0.2, 0.1, 1.8],
               [0.9, 0.8, 1.0, 0.7],
               [2.5, -0.5, 0.0, 0.3]])
yb = np.array([0, 1, 3, 2, 1])


### Exercise 11.5 (In class)

Compute the average softmax cross-entropy for a batch of 5 examples with 4 classes, written with numpy only. Work from the scores with the log-sum-exp trick: for each row, the loss is $\log\sum_j e^{z_j} - z_{\text{true}}$ (subtract the row maximum before exponentiating). Do not take the log of a softmax.

- Inputs: `Zb` (array of shape (5, 4), the scores) and `yb` (the index of the true class for each of the 5 examples).
- Define: `ce_batch`, one number: the average over the 5 examples of $-\log p$ of the true class.


In [ ]:
# Exercise 11.5: define ce_batch
ce_batch = None


In [ ]:
check("ce_batch", ce_batch, "814bd26f2f7b6166")


## Exit problem (on paper, 10 minutes)

**Part A.** Four polynomial models were fitted to the same data. Training error and validation error (mean squared error) are:

| Model | Training error | Validation error |
|---|---|---|
| A (degree 1) | 4.1 | 4.3 |
| B (degree 3) | 2.0 | 2.2 |
| C (degree 8) | 0.9 | 3.5 |
| D (degree 20) | 0.1 | 9.0 |

Which model do you pick? Say what you look at, and explain in two sentences what is wrong with the models you did not pick.

**Part B.** You fit a ridge regression. What happens to the coefficients as $\lambda$ grows? Why must the features be on one scale first? Where do the mean and standard deviation used for scaling come from?


## At home

### Exercise 11.6 setup: ridge by gradient descent, and why scaling helps

Networks are trained by gradient descent, not by a closed form. Ridge is the smallest example where we can check gradient descent against the exact answer. We use the German credit data: the outcome is the log loan amount (centered) and the features are `duration`, `installment`, `residence`, `age`, `cards`, `liable`.

The loss here uses the *mean* squared error (a mean over the $N$ rows, not a sum), with a penalty on each coefficient:

$$
L(\beta) = \frac{1}{N}\lVert y - X\beta\rVert^2 + \sum_j \rho_j \beta_j^2, \qquad \nabla L = -\frac{2}{N} X^\top (y - X\beta) + 2\,\rho \odot \beta .
$$

Here $\rho_j$ is the penalty on coefficient $j$ (for ordinary ridge, all $\rho_j = \lambda$). With a sum instead of a mean the matching penalty is $N$ times larger.

Raw and standardized ridge are **different penalized problems**. If $s_j$ is the standard deviation of feature $j$, the standardized coefficient is $\gamma_j = s_j \beta_j$ (so that predictions agree). A penalty $\lambda\gamma_j^2$ on the standardized scale is $\lambda s_j^2 \beta_j^2$ on the raw scale. So plain $\lambda\beta_j^2$ on raw features is not the same fit. We compare two versions of the same objective, with $\lambda = 0.1$:

- A: standardized features, penalty $\lambda$.
- B: raw features, penalty $\lambda s_j^2$. This is the same objective as A, written in raw coordinates. Its predictions must agree with A.

Gradient descent repeats $\beta \leftarrow \beta - \eta \nabla L(\beta)$ from $\beta = 0$, with the step $\eta = 1/\ell$, where $\ell$ is the largest eigenvalue of the Hessian $H = \tfrac{2}{N} X^\top X + 2\,\mathrm{diag}(\rho)$. (The largest stable step is $2/\ell$; we use half of that.) Stop at the first step count `t` for which $\lVert\nabla L(\beta_t)\rVert < 10^{-6}$.


In [ ]:
cols = ["duration", "installment", "residence", "age", "cards", "liable"]
Xc = credit[cols].to_numpy(float)
yc = np.log(credit["amount"].to_numpy(float))

lam_gd = 0.1
yg = yc - yc.mean()
X_raw = Xc - Xc.mean(axis=0)                      # centered, in the original units
sd_x = Xc.std(axis=0)
X_std = X_raw / sd_x                              # centered and divided by the sd
N = len(yg)
pen_A = np.full(6, lam_gd)                        # penalty for A (standardized features)
pen_B = lam_gd * sd_x**2                          # penalty for B (raw features, same objective)
print("sd of the six features:", sd_x.round(2))


### Exercise 11.6 (At home)

Fill in the gradient descent update, run it for A and for B, and report the step counts and how close the two fits are.

- Inputs: `X_std`, `X_raw`, `yg`, `pen_A`, `pen_B` (defined above). Start at $\beta = 0$, step $\eta = 1/\ell$ with $\ell$ the largest eigenvalue of $H = \tfrac{2}{N} X^\top X + 2\,\mathrm{diag}(\rho)$ (`np.linalg.eigvalsh(H).max()`), stop at the first step count `t` with $\lVert\nabla L\rVert < 10^{-6}$ (`t = 0` means the start already passes).
- Define: `steps_A` and `steps_B` (integers, the step counts for A on `X_std` and B on `X_raw`), and `gap_AB`: the largest absolute difference between the predictions `X_std @ beta_A` and `X_raw @ beta_B` (must be below 0.0001).


In [ ]:
# Exercise 11.6: define steps_A, steps_B, gap_AB
def gradient_descent(X, y, pen, tol=1e-6, max_steps=200_000):
    N_ = len(y)
    H = (2 / N_) * X.T @ X + 2 * np.diag(pen)
    eta = 1 / np.linalg.eigvalsh(H).max()
    beta = np.zeros(X.shape[1])
    for t in range(max_steps):
        grad = -(2 / N_) * X.T @ (y - X @ beta) + 2 * pen * beta
        if np.linalg.norm(grad) < tol:
            return t, beta
        beta = beta                                # fill in: the gradient descent update
    return max_steps, beta

steps_A = steps_B = gap_AB = None


In [ ]:
check("steps_A", steps_A, "cc4448973f430034")
check("steps_B", steps_B, "04fad24d68b9bb47")
assert gap_AB is not None, 'gap_AB is missing: run the exercise cell above and define gap_AB'
assert 0 <= gap_AB < 1e-4, f"gap_AB = {gap_AB:.2e}: A and B should give the same predictions (below 0.0001)"
print("ok  gap_AB")


In [ ]:
# reference run (does not use your cell above), plus version C: raw features with the plain penalty
def gd_ref(X, y, pen, tol=1e-6, max_steps=200_000):
    N_ = len(y)
    H = (2 / N_) * X.T @ X + 2 * np.diag(pen)
    eta = 1 / np.linalg.eigvalsh(H).max()
    beta = np.zeros(X.shape[1])
    for t in range(max_steps):
        grad = -(2 / N_) * X.T @ (y - X @ beta) + 2 * pen * beta
        if np.linalg.norm(grad) < tol:
            return t, beta
        beta = beta - eta * grad
    return max_steps, beta

def closed_form(X, y, pen):
    return np.linalg.solve((2 / len(y)) * X.T @ X + 2 * np.diag(pen), (2 / len(y)) * X.T @ y)

refA, bA = gd_ref(X_std, yg, pen_A)
refB, bB = gd_ref(X_raw, yg, pen_B)
refC, bC = gd_ref(X_raw, yg, np.full(6, lam_gd))
gap_AC = np.max(np.abs(X_std @ bA - X_raw @ bC))
err_A = np.max(np.abs(bA - closed_form(X_std, yg, pen_A)))
err_B = np.max(np.abs(bB - closed_form(X_raw, yg, pen_B)))
cond_A = np.linalg.cond((2 / N) * X_std.T @ X_std + 2 * np.diag(pen_A))
cond_B = np.linalg.cond((2 / N) * X_raw.T @ X_raw + 2 * np.diag(pen_B))
print("steps:  A", refA, "  B", refB, "  C (raw, plain penalty)", refC)
print("largest gap between gradient descent and its own closed form:  A", f"{err_A:.1e}", "  B", f"{err_B:.1e}")
print("largest difference in predictions:  A against C", round(gap_AC, 4))
print("condition number of the Hessian:  A", round(cond_A, 2), "  B", round(cond_B, 1))


**Reading.**

- A (standardized) converges in 17 steps. B (raw coordinates, same objective) needs 10542 steps, about 620 times more. Version C (raw, plain penalty) needs 5809.
- Each run agrees with its own closed form (largest gap 2.2e-07 for A and 3.6e-06 for B).
- A and B give the same predictions: the same statistical fit, reached at very different cost. Version C is a different fit: its predictions differ from A by up to 0.2075.
- The condition number of the Hessian is 1.87 for A and 1152 for B. A large condition number means the loss surface is a long narrow valley: steep across, flat along. The speed of gradient descent is roughly governed by this number: with one step size it zigzags across the valley and creeps along it.
- These counts are results for this dataset, this step size and this stopping rule. The lesson in one sentence: scaling can change optimization speed, while changing the penalty can change the fitted model.
- This is why networks normalize their inputs. Batch normalization applies the same idea to the inputs of every layer.
